# ISMIP7 Greenland: regional mass balance against Mankoff

The heavy lifting lives in `pism_terra.ismip7.greenland.mass_balance`, the module behind
`pism-ismip7-greenland-mass-balance`. This notebook walks through its steps so you can look at the
intermediate objects: the lazy ensemble on `(gcm_id, ssp_id)`, the basin integrals, the observations.

**Inputs** are taken from the widgets below. The root is a run's `output` directory, either in the
bucket (read anonymously; a `*` in place of the job id collects a cloud project's per-job directories)
or a local copy of the same tree.

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib as mpl
import matplotlib.pylab as plt
import numpy as np
import xarray as xr
from IPython.display import Image, display

from pism_terra.ismip7.greenland import mass_balance as mb
from pism_terra.plotting import rc_params

#: Basins in plotting order, the ice sheet first.
ORDER = ["GIS_GIS", "GIS_NO", "GIS_NE", "GIS_CE", "GIS_SE", "GIS_SW", "GIS_CW", "GIS_NW"]

## Inputs

Edit the fields, then run the cells below. The two example roots are the cloud project and a local copy of it.

In [ ]:
root_w = widgets.Text(
    value="s3://pism-cloud-data/ismip7_production/2026_09_core/*/output",
    placeholder="s3://bucket/name/project/*/output  or  2026_10_ismip7_core_ctrl/output",
    description="Root",
    layout=widgets.Layout(width="90%"),
)
output_w = widgets.Text(value="figures/ismip7", description="Output dir", layout=widgets.Layout(width="60%"))
variables_w = widgets.Text(value=",".join(mb.DEFAULT_VARIABLES), description="Fluxes", layout=widgets.Layout(width="60%"))
reference_w = widgets.Text(value=mb.DEFAULT_REFERENCE_YEAR, description="Reference year")
sigma_w = widgets.FloatText(value=1.0, description="Obs. band (σ)")
xlim_w = widgets.Text(value="1985,2100", description="Years shown")
workers_w = widgets.IntText(value=0, description="Dask workers", tooltip="0 runs on the threaded scheduler")
display(widgets.VBox([root_w, output_w, variables_w, reference_w, sigma_w, xlim_w, workers_w]))

In [ ]:
ROOT = root_w.value.strip()
OUTPUT = Path(output_w.value.strip())
OUTPUT.mkdir(parents=True, exist_ok=True)
VARIABLES = [v.strip() for v in variables_w.value.split(",") if v.strip()]
REFERENCE_YEAR = reference_w.value.strip()
SIGMA = float(sigma_w.value)
XLIM = tuple(y.strip() for y in xlim_w.value.split(","))

client = None
if workers_w.value > 0:
    from dask.distributed import Client

    client = Client(n_workers=workers_w.value)
    print(client.dashboard_link)
print(f"root {ROOT}\noutput {OUTPUT.resolve()}\nfluxes {VARIABLES}, reference year {REFERENCE_YEAR}")

## The ensemble

One file per variable and counter, opened lazily as `(gcm_id, ssp_id, time, y, x)`. Files a run in
flight has left empty are skipped with a warning.

In [ ]:
paths = mb.find_files(ROOT, VARIABLES)
ds = mb.open_submission(paths)
ds

## Basin integrals

The one pass over the data: every flux contracted with the stacked basin masks, converted to Gt/yr,
summed into a mass balance and accumulated from the reference year. The result is small and kept.

In [ ]:
outline = mb.resolve_outline(ROOT, None)
regions = mb.compute_regions(ds, outline, variables=VARIABLES, reference_year=REFERENCE_YEAR)
regions = regions.reindex(region=[r for r in ORDER if r in regions["region"].values])
regions.to_netcdf(OUTPUT / "regional_mass_balance.nc")
regions[["mass_balance", "cumulative_mass_balance"]].to_dataframe().to_csv(OUTPUT / "regional_mass_balance.csv")
regions

## Observations

The staged Mankoff et al. (2021) product, on annual bins and zeroed at the same reference year, with the
flux series kept for the panels further down.

In [ ]:
mankoff_url = mb.first_match(ROOT, f"observations/{mb.DEFAULT_MANKOFF}")
mankoff = mb.load_mankoff(
    mankoff_url, reference_year=REFERENCE_YEAR, variables=mb.MANKOFF_CUMULATIVE + mb.MANKOFF_FLUXES
)
mankoff

## Cumulative mass balance per basin

In [ ]:
overview = OUTPUT / "regional_mass_balance.png"
mb.plot_regions(regions, mankoff, overview, sigma=SIGMA, xlim=XLIM)
display(Image(filename=overview))

## Fluxes per basin

Mass balance, surface mass balance and grounding-line flux of every (GCM, pathway) against the observed
series and its band.

In [ ]:
PANELS = [
    ("mass_balance", "mass_balance", "Mass balance"),
    ("acabf", "surface_mass_balance", "Surface mass balance"),
    ("ligroundf", "grounding_line_flux", "Grounding line flux"),
]


def plot_fluxes(region: str, filename: Path) -> None:
    """Three flux panels for one basin: model lines over the observed band."""
    model = regions.sel(region=region)
    obs = mankoff.sel(region=region)
    with mpl.rc_context(rc=dict(rc_params, **{"font.size": 5})):
        fig, axs = plt.subplots(len(PANELS), 1, figsize=(6.2, 4.2), sharex=True, layout="constrained")
        for ax, (sim_var, obs_var, label) in zip(axs, PANELS):
            ax.fill_between(
                obs["time"].values,
                obs[obs_var] - SIGMA * obs[f"{obs_var}_uncertainty"],
                obs[obs_var] + SIGMA * obs[f"{obs_var}_uncertainty"],
                lw=0, color="0.75", alpha=0.5,
            )
            ax.plot(obs["time"].values, obs[obs_var], lw=1, color="k", label="Mankoff")
            if sim_var in model:
                for gcm in model["gcm_id"].values:
                    for ssp in model["ssp_id"].values:
                        line = model[sim_var].sel(gcm_id=gcm, ssp_id=ssp).dropna("time")
                        if line.size:
                            ax.plot(line["time"].values, line, lw=0.6, ls=mb.GCM_STYLES.get(str(gcm), "solid"),
                                    color=mb.SSP_COLORS.get(str(ssp), "0.3"), label=f"{gcm} {ssp}")
            ax.set_ylabel(f"{label}\n({mb.FLUX_UNITS})")
        axs[0].legend(fontsize=4, frameon=False, ncol=3)
        axs[-1].set_xlim(np.datetime64(XLIM[0]), np.datetime64(XLIM[1]))
        fig.suptitle(region, fontsize=6)
        fig.savefig(filename, dpi=300)
        plt.close(fig)


for region in regions["region"].values:
    plot_fluxes(str(region), OUTPUT / f"fluxes_{region}.png")
display(Image(filename=OUTPUT / "fluxes_GIS_GIS.png"))

## Later sessions

The basin series are saved, so a plot can be redone without touching the ensemble:

In [ ]:
# regions = xr.open_dataset(OUTPUT / "regional_mass_balance.nc").load()
# mb.plot_regions(regions, mankoff, OUTPUT / "regional_mass_balance.png", sigma=SIGMA, xlim=XLIM)